# Scraping Data Berita Sport & Finance (Detik.com)

Berisi dokumentasi dan implementasi dari proses *crawling* serta *scraping* artikel berita portal daring **Detik.com** untuk dua kategori:
1. **Kategori Sport (Sub-kategori Bulutangkis/Raket)**: 100 artikel berita (ID: 1 - 100)
2. **Kategori Finance (Sub-kategori Finansial)**: 100 artikel berita (ID: 101 - 200)

Total dataset yang dikumpulkan berjumlah **200 artikel**, disimpan ke dalam format tabular dengan atribut `ID`, `Isi berita`, dan `label`.

## 1. Import Library Pendukung

Import Library Python yang akan digunakan dalam proses ekstraksi dan pengolahan data:
- **`trafilatura`**: Pustaka khusus penambangan web (*web scraping*) yang dirancang untuk mengunduh (*fetching*) konten web dan mengekstraksi teks utama artikel berita dengan memfilter elemen *boilerplate* (iklan, menu navigasi, header, dan footer) secara otomatis.
- **`pandas` (`pd`)**: Pustaka untuk pengolahan dan manipulasi data berbasis tabel (*DataFrame*), digunakan untuk menyusun dataset hasil ekstraksi dan menggabungkan data antar-kategori.
- **`lxml.html`**: Modul parser HTML berbasis library C `libxml2` yang cepat dan efisien. Digunakan untuk mem-parsing string HTML menjadi struktur pohon elemen (*DOM tree*) sehingga atribut tag (seperti hyperlink `href`) dapat diseleksi dengan ekspresi **XPath**.

In [1]:
import trafilatura
import pandas as pd
from lxml import html

## 2. Scraping Data Kategori Sport (Raket)

### 2.1 Mengakses Halaman Utama Kanal Raket & Ekstraksi Link Awal
Berfungsi untuk mengakses halaman awal website olahraga bulutangkis Detik Sport dan mengekstrak semua hyperlink awal:
1. **Inisialisasi URL**: Menentukan target halaman awal pada `https://sport.detik.com/raket`.
2. **Mengunduh Halaman Web**: Fungsi `trafilatura.fetch_url(url)` mengirimkan HTTP request untuk mengambil seluruh kode sumber HTML dari halaman web tersebut.
3. **Membangun Struktur Tree HTML**: Fungsi `html.fromstring(downloaded)` mengonversi raw HTML string menjadi representasi hierarki elemen (*DOM tree*).
4. **Ekstraksi Tautan dengan XPath**: Ekspresi `tree.xpath('//a/@href')` mengekstrak seluruh nilai atribut `href` dari seluruh tag tautan (`<a>`) yang terdapat pada halaman.

In [2]:
url = "https://sport.detik.com/raket"

downloaded = trafilatura.fetch_url(url)
tree = html.fromstring(downloaded)

links = tree.xpath('//a/@href')

### 2.2 Menyaring Tautan Khusus Artikel Berita Sport
Menyaring tautan yang terkumpul agar hanya tautan artikel berita yang disimpan:
- Pada kanal Detik Sport Raket, URL artikel berita memiliki pola pengenal khusus yaitu mengandung substring `"/raket/d-"`.
- Dilakukan perulangan (`for link in links`) untuk memeriksa setiap URL. Hanya tautan yang memuat pola tersebut yang dimasukkan ke dalam list `article_links`, mengabaikan tautan menu, kategori lain, atau link media sosial.

In [3]:
article_links = []

for link in links:
    if "/raket/d-" in link:
        article_links.append(link)

### 2.3 Menghapus Tautan Duplikat (Deduplikasi)
Untuk memastikan bahwa kumpulan tautan artikel tidak memiliki URL ganda:
- Dalam satu halaman berita, sering kali artikel yang sama ditautkan beberapa kali (misalnya pada gambar/thumbnail, judul utama, dan daftar berita populer).
- Fungsi `dict.fromkeys(article_links)` menghapus duplikasi tautan secara cepat sekaligus menjaga urutan kemunculan tautan (*order-preserving*).
- Hasilnya diubah kembali menjadi list `all_article_links` dan jumlah tautan unik yang didapatkan dari halaman utama ditampilkan.

In [4]:
all_article_links = list(dict.fromkeys(article_links))

print("Dari halaman utama:", len(all_article_links))

Dari halaman utama: 51


### 2.4 Scraping Halaman Indeks untuk Memenuhi Target 100 Artikel Sport
Melakukan penelusuran Pagination pada halaman indeks berita sport hingga kuota 100 artikel terpenuhi:
1. **Perulangan Halaman Indeks**: Mengiterasi halaman indeks dengan URL berparameter `https://sport.detik.com/raket/indeks?page={page}`.
2. **Kondisi Berhenti (*Early Exit*)**: Mengecek apakah jumlah tautan unik pada `all_article_links` telah mencapai minimal 100. Jika ya, perulangan langsung dihentikan (`break`).
3. **Download & Ekstraksi XPath**: Setiap halaman indeks diunduh dan diparsing untuk mengambil seluruh hyperlink menggunakan XPath `//a/@href`.
4. **Validasi Tautan Baru**: Tautan dicek apakah berformat artikel (`"/raket/d-"`) dan belum pernah ada di `all_article_links` sebelum ditambahkan.
5. **Pencatatan Progres**: Menampilkan akumulasi jumlah tautan pada setiap halaman indeks yang diperiksa.

In [5]:
for page in range(1, 157):

    if len(all_article_links) >= 100:
        break

    url_index = f"https://sport.detik.com/raket/indeks?page={page}"

    downloaded = trafilatura.fetch_url(url_index)

    if downloaded is None:
        continue

    tree = html.fromstring(downloaded)
    links = tree.xpath('//a/@href')

    for link in links:
        if "/raket/d-" in link and link not in all_article_links:
            all_article_links.append(link)

    print("Page", page, "-> total:", len(all_article_links))


Page 1 -> total: 51
Page 2 -> total: 51
Page 3 -> total: 60
Page 4 -> total: 80
Page 5 -> total: 100


### 2.5 Pembatasan Tepat 100 Tautan Berita Sport
Memastikan jumlah tautan yang akan diekstraksi isinya tepat berjumlah 100:
- Menggunakan teknik *slicing* `[:100]` pada list `all_article_links` untuk mengambil 100 URL pertama dan menyimpannya pada variabel `sport_links`.
- Mencetak jumlah total tautan untuk memvalidasi kesiapan tahap ekstraksi konten.

In [6]:
sport_links = all_article_links[:100]

print("total berita sport:", len(sport_links))


total berita sport: 100


### 2.6 Ekstraksi Konten dan Teks Utama Berita Sport
Berfungsi mengunduh halaman tiap artikel secara berurutan dan mengekstrak isi teks beritanya:
1. **Ekstraksi Teks Bersih**: Untuk setiap URL pada `sport_links`, fungsi `trafilatura.fetch_url(url)` mengambil halaman artikel, lalu `trafilatura.extract(downloaded)` mengekstrak paragraf berita utama tanpa menyertakan elemen iklan, navigasi, maupun komentar.
2. **Penyimpanan Terstruktur**: Setiap data artikel disimpan dalam bentuk dictionary ke list `sport_data` dengan struktur:
   - `ID`: Nomor urut artikel (1 s.d. 100).
   - `Isi berita`: Teks isi artikel berita hasil ekstraksi.
   - `label`: Kategori data (`"Sport"`).
3. **Etika Scraping (*Delay*)**: Perintah `time.sleep(1)` memberikan jeda waktu 1 detik antar-request guna menjaga performa server (*politeness policy*) dan meminimalkan risiko pembatasan akses (*rate limit / IP blocking*).
4. **Penanganan Kesalahan (*Error Handling*)**: Menggunakan blok `try-except` agar jika terjadi kendala pada satu URL, proses scraping tetap berlanjut ke URL berikutnya.

In [7]:
import time

sport_data = []

for i, url in enumerate(sport_links, start=1):
    try:
        downloaded = trafilatura.fetch_url(url)
        if downloaded:
            text = trafilatura.extract(downloaded)
            if text:
                sport_data.append({
                    "ID": i,
                    "Isi berita": text,
                    "label": "Sport"
                })
        print(f"{i}/100 selesai")
        time.sleep(1)
    except Exception as e:
        print(f"{i}/100 gagal: {e}")

print("Jumlah berita berhasil:", len(sport_data))

1/100 selesai
2/100 selesai
3/100 selesai
4/100 selesai
5/100 selesai
6/100 selesai
7/100 selesai
8/100 selesai
9/100 selesai
10/100 selesai
11/100 selesai
12/100 selesai
13/100 selesai
14/100 selesai
15/100 selesai
16/100 selesai
17/100 selesai
18/100 selesai
19/100 selesai
20/100 selesai
21/100 selesai
22/100 selesai
23/100 selesai
24/100 selesai
25/100 selesai
26/100 selesai
27/100 selesai
28/100 selesai
29/100 selesai
30/100 selesai
31/100 selesai
32/100 selesai
33/100 selesai
34/100 selesai
35/100 selesai
36/100 selesai
37/100 selesai
38/100 selesai
39/100 selesai
40/100 selesai
41/100 selesai
42/100 selesai
43/100 selesai
44/100 selesai
45/100 selesai
46/100 selesai
47/100 selesai
48/100 selesai
49/100 selesai
50/100 selesai
51/100 selesai
52/100 selesai
53/100 selesai
54/100 selesai
55/100 selesai
56/100 selesai
57/100 selesai
58/100 selesai
59/100 selesai
60/100 selesai
61/100 selesai
62/100 selesai
63/100 selesai
64/100 selesai
65/100 selesai
66/100 selesai
67/100 selesai
68/1

### 2.7 Pembuatan DataFrame Kategori Sport
Lonversi kumpulan data berita sport menjadi objek tabel terstruktur:
- `pd.DataFrame(sport_data)`: Mengubah list dictionary `sport_data` menjadi Pandas DataFrame (`df_sport`).
- `df_sport.head()`: Menampilkan 5 baris pertama data untuk memverifikasi kesesuaian kolom `ID`, `Isi berita`, dan `label`.

In [8]:
df_sport = pd.DataFrame(sport_data)

df_sport.head()

,ID,Isi berita,label
0,1,Putri Kusuma Wardani percaya diri dengan kompo...,Sport
1,2,Momen debut di Asian Games 2026 tak mau disia-...,Sport
2,3,Sektor ganda campuran kembali mengalami peromb...,Sport
3,4,Nova Widianto mulai menangani tim ganda campur...,Sport
4,5,Seri V Men's World Tennis Championship 2026 su...,Sport


## 3. Scraping Data Kategori Finance

### 3.1 Mengumpulkan 100 Tautan Berita Finance
Cell ini berfungsi mengumpulkan 100 tautan unik artikel berita untuk kategori finansial (Detik Finance):
1. **Pengambilan dari Halaman Utama**: Mengunduh halaman `https://finance.detik.com/finansial`, mengekstrak semua hyperlink, dan menyaring tautan yang berawalan `"https://finance.detik.com/"` serta mengandung pola artikel `"/d-"`.
2. **Penelusuran Halaman Indeks**: Melakukan iterasi halaman indeks `https://finance.detik.com/indeks?page={page}` untuk menambah tautan baru yang unik hingga target tercapai.
3. **Pemilihan 100 Data**: Memotong list dengan *slicing* `finance_article_links[:100]` ke dalam variabel `finance_links` dan mencetak jumlah akhirnya.

In [9]:
# Mulai dari halaman utama Finance
url = "https://finance.detik.com/finansial"

downloaded = trafilatura.fetch_url(url)
tree = html.fromstring(downloaded)
links = tree.xpath('//a/@href')

finance_article_links = []

for link in links:
    if link.startswith("https://finance.detik.com/") and "/d-" in link:
        if link not in finance_article_links:
            finance_article_links.append(link)

print("Dari halaman utama:", len(finance_article_links))


# Tambahkan dari halaman indeks
for page in range(1, 20):

    if len(finance_article_links) >= 100:
        break

    url_index = f"https://finance.detik.com/indeks?page={page}"

    downloaded = trafilatura.fetch_url(url_index)

    if downloaded is None:
        continue

    tree = html.fromstring(downloaded)
    links = tree.xpath('//a/@href')

    for link in links:
        if link.startswith("https://finance.detik.com/") and "/d-" in link:
            if link not in finance_article_links:
                finance_article_links.append(link)

    print("Page", page, "-> total:", len(finance_article_links))


# Ambil tepat 100 artikel
finance_links = finance_article_links[:100]

print()
print("================================")
print("TOTAL BERITA FINANCE:", len(finance_links))
print("================================")

Dari halaman utama: 72
Page 1 -> total: 85
Page 2 -> total: 102

TOTAL BERITA FINANCE: 100


### 3.2 Ekstraksi Konten dan Teks Utama Berita Finance
Mengunduh dan mengekstrak isi teks berita kategori finance dari 100 URL yang telah didapatkan:
1. **Penomoran Kontinu**: Parameter `enumerate(finance_links, start=101)` memastikan `ID` artikel bernomor `101` sampai `200` agar tetap berurutan dan unik ketika digabungkan dengan kategori Sport.
2. **Ekstraksi Teks**: Menggunakan `trafilatura.extract()` untuk memperoleh teks berita yang bersih.
3. **Pelabelan**: Memberi label `"Finance"` pada setiap artikel berita.
4. **Jeda Waktu (*Delay*)**: Memberikan jeda 1 detik (`time.sleep(1)`) di setiap iterasi untuk menjaga stabilitas request.
5. **Logging Progress**: Menampilkan indikator selesai atau gagal untuk setiap nomor berita secara bertahap.

In [14]:
import time

finance_data = []

for i, url in enumerate(finance_links, start=101):
    try:
        downloaded = trafilatura.fetch_url(url)

        if downloaded:
            text = trafilatura.extract(downloaded)

            if text:
                finance_data.append({
                    "ID": i,
                    "Isi berita": text,
                    "label": "Finance"
                })

        print(f"{i}/200 selesai")

        time.sleep(1)

    except Exception as e:
        print(f"{i}/200 gagal: {e}")

print()
print("Jumlah berita Finance berhasil:", len(finance_data))

101/200 selesai
102/200 selesai
103/200 selesai
104/200 selesai
105/200 selesai
106/200 selesai
107/200 selesai
108/200 selesai
109/200 selesai
110/200 selesai
111/200 selesai
112/200 selesai
113/200 selesai
114/200 selesai
115/200 selesai
116/200 selesai
117/200 selesai
118/200 selesai
119/200 selesai
120/200 selesai
121/200 selesai
122/200 selesai
123/200 selesai
124/200 selesai
125/200 selesai
126/200 selesai
127/200 selesai
128/200 selesai
129/200 selesai
130/200 selesai
131/200 selesai
132/200 selesai
133/200 selesai
134/200 selesai
135/200 selesai
136/200 selesai
137/200 selesai
138/200 selesai
139/200 selesai
140/200 selesai
141/200 selesai
142/200 selesai
143/200 selesai
144/200 selesai
145/200 selesai
146/200 selesai
147/200 selesai
148/200 selesai
149/200 selesai
150/200 selesai
151/200 selesai
152/200 selesai
153/200 selesai
154/200 selesai
155/200 selesai
156/200 selesai
157/200 selesai
158/200 selesai
159/200 selesai
160/200 selesai
161/200 selesai
162/200 selesai
163/200 

### 3.3 Pembuatan DataFrame Kategori Finance
Cell ini berfungsi membentuk tabel data terstruktur untuk kategori finance:
- `pd.DataFrame(finance_data)`: Mengonversi data list dictionary `finance_data` menjadi Pandas DataFrame (`df_finance`).
- `df_finance.head()`: Menampilkan 5 sampel data pertama untuk memeriksa kebenaran data hasil scraping.

In [15]:
df_finance = pd.DataFrame(finance_data)

df_finance.head()

,ID,Isi berita,label
0,101,Mahkamah Agung (MA) resmi melantik Sarjito seb...,Finance
1,102,Peraturan Otoritas Jasa Keuangan (POJK) demutu...,Finance
2,103,PT Swayasa Prakasa Tbk (SWAP) dikabarkan menun...,Finance
3,104,PT Jasa Marga (Persero) Tbk (JSMR) terbuka unt...,Finance
4,105,PT Rukun Raharja Tbk (RAJA) telah merampungkan...,Finance


## 4. Penggabungan Dataset & Validasi Akhir

### 4.1 Menggabungkan Dataset Sport dan Finance
Cell ini satukan kedua dataset kategori menjadi satu dataset akhir yang utuh:
- `pd.concat([df_sport, df_finance], ignore_index=True)`: Menggabungkan baris dari `df_sport` (100 data) dan `df_finance` (100 data) secara vertikal, sekaligus menata ulang nomor indeks baris dari 0 hingga 199.
- `df_final.head()`: Menampilkan sampel data awal dari dataset gabungan.
- `print("Jumlah data:", len(df_final))`: Memverifikasi bahwa total data yang terkumpul telah genap mencapai **200 data artikel berita**.

In [16]:
df_final = pd.concat([df_sport, df_finance], ignore_index=True)


df_final.head()
print("Jumlah data:", len(df_final))

Jumlah data: 200


### 4.2 Menampilkan 5 Data Awal dan 5 Data Akhir Dataset Gabungan
Cell ini berfungsi untuk menampilkan sampel data dari hasil penggabungan kedua kategori:
- **`df_final.head()`**: Menampilkan 5 baris pertama data yang mewakili kategori **Sport** (indeks 0 s.d. 4).
- **`df_final.tail()`**: Menampilkan 5 baris terakhir data yang mewakili kategori **Finance** (indeks 195 s.d. 199).
- Menggunakan fungsi `display()` agar kedua tabel DataFrame (awal dan akhir) dapat ditampilkan secara bersamaan dalam satu cell.

In [17]:
# Menampilkan 5 data awal (Kategori Sport)
display(df_final.head())

# Menampilkan 5 data akhir (Kategori Finance)
display(df_final.tail())

,ID,Isi berita,label
0,1,Putri Kusuma Wardani percaya diri dengan kompo...,Sport
1,2,Momen debut di Asian Games 2026 tak mau disia-...,Sport
2,3,Sektor ganda campuran kembali mengalami peromb...,Sport
3,4,Nova Widianto mulai menangani tim ganda campur...,Sport
4,5,Seri V Men's World Tennis Championship 2026 su...,Sport


,ID,Isi berita,label
195,196,Buat kamu yang baru masuk kerja jadi staf admi...,Finance
196,197,KAI Properti melakukan pemeliharaan Listrik Al...,Finance
197,198,Kebijakan mengolah sampah menjadi energi yang ...,Finance
198,199,Punya sertifikat Brevet Pajak A dan B memang j...,Finance
199,200,Bisnis yang beroperasi lintas negara punya sat...,Finance


## 5. Pembersihan Teks & Ekspor Dataset ke CSV / Excel

### 5.1 Menghapus Karakter Baris Baru (*Newline*)
Ketika teks berita diekstraksi langsung dari halaman web, konten artikel memuat banyak karakter pemisah paragraf (`\n` atau `\r`). Jika langsung diekspor ke format CSV mentah tanpa dinormalisasi, aplikasi *spreadsheet* (seperti Microsoft Excel) akan membaca *line break* tersebut sebagai baris bertumpuk atau merusak struktur baris.

Oleh karena itu, dilakukan pra-pemrosesan sederhana untuk:
1. Mengganti seluruh karakter baris baru (`\n`, `\r`, `\t`) dengan spasi tunggal.
2. Menghapus spasi ganda berturut-turut agar teks rapi dan padat.
3. Memastikan setiap 1 artikel berita menempati tepat **1 baris data** dalam file CSV/Excel.

In [18]:
import re

# Fungsi untuk membersihkan newline dan spasi berlebih
def bersihkan_teks(teks):
    if not isinstance(teks, str):
        return ""
    # Ganti newline (\n), carriage return (\r), dan tab (\t) menjadi spasi
    teks = re.sub(r"[\r\n\t]+", " ", teks)
    # Hapus spasi ganda berlebih
    teks = re.sub(r"\s+", " ", teks)
    return teks.strip()

# Terapkan pembersihan pada kolom 'Isi berita'
df_final["Isi berita"] = df_final["Isi berita"].apply(bersihkan_teks)

# Menampilkan sampel 3 data teratas setelah dibersihkan
df_final.head(3)

### 5.2 Menyimpan Dataset ke Format CSV dan Excel (.xlsx)
Menyimpan dataset gabungan yang telah bersih dan rapi:
- **`dataset_berita_detik.csv`**: Menggunakan parameter `encoding="utf-8-sig"` agar dikenali dengan baik oleh Microsoft Excel di Windows tanpa masalah encoding/karakter rusak.
- **`dataset_berita_detik.xlsx`**: File format Excel asli untuk kemudahan visualisasi langsung dengan kolom yang tertata otomatis.

In [ ]:
# 1. Simpan ke CSV dengan encoding UTF-8 with BOM (Excel-friendly)
# df_final.to_csv("dataset_berita_detik.csv", index=False, encoding="utf-8-sig")

# 2. Simpan ke Excel (.xlsx)
df_final.to_excel("dataset_berita_detik.xlsx", index=False)

print("Dataset berhasil disimpan dengan rapi:")
print(f"- Total baris data : {len(df_final)} artikel")
# print("- File CSV         : dataset_berita_detik.csv")
print("- File Excel       : dataset_berita_detik.xlsx")

Dataset berhasil disimpan dengan rapi:
- Total baris data : 200 artikel
- File CSV         : dataset_berita_detik.csv
- File Excel       : dataset_berita_detik.xlsx
